# Gradio Application

## Set-Up

In [1]:
import pandas as pd
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("application").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/29 00:38:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel, GBTClassificationModel
import pickle

# TODO: Get pipelines to feed data through
tabular_pipeline = PipelineModel.load("hdfs://localhost:9000/user/jj/final_proj/tabular_pipeline.parquet")
lr_model = LogisticRegressionModel.load("hdfs://localhost:9000/user/jj/final_proj/lr_model")
rf_model = RandomForestClassificationModel.load("hdfs://localhost:9000/user/jj/final_proj/rf_model")
gbt_model = GBTClassificationModel.load("hdfs://localhost:9000/user/jj/final_proj/gbt_model")
image_rf_model = RandomForestClassificationModel.load("hdfs://localhost:9000/user/jj/final_proj/images_rf_model")
with open("images_pca_model.pkl", "rb") as f:
    images_pca_model = pickle.load(f)

25/04/29 00:38:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

## Creating Prediction Methods

In [10]:
def recode_inputs(gender, race, smoker, chf, chd, heart_attack, stroke, thyroid):
    gender_code = 1.0 if gender == "Male" else 2.0

    race_map = {
        "Mexican American": 1.0,
        "Other Hispanic": 2.0,
        "Non-Hispanic White": 3.0,
        "Non-Hispanic Black": 4.0,
        "Non-Hispanic Asian": 6.0,
        "Other/Multi-Racial": 7.0
    }
    race_code = race_map[race]

    yes_no_map = {
        "Yes": 1.0,
        "No": 2.0
    }
    smoker_code = yes_no_map[smoker]
    chf_code = yes_no_map[chf]
    chd_code = yes_no_map[chd]
    heart_attack_code = yes_no_map[heart_attack]
    stroke_code = yes_no_map[stroke]
    thyroid_code = yes_no_map[thyroid]

    return gender_code, race_code, smoker_code, chf_code, chd_code, heart_attack_code, stroke_code, thyroid_code


In [15]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

threshold = 0.91

extract_prob_1 = F.udf(lambda v: float(v[1]), DoubleType())


def predict_from_tabular(gender, age, race, bmi, alcohol, smoker, chf, chd, heart_attack, stroke, thyroid):
    # Recode inputs as they were in the initial data
    gender, race, smoker, chf, chd, heart_attack, stroke, thyroid = recode_inputs(gender, race, smoker, chf, chd, heart_attack, stroke, thyroid)

    
    # Convert inputs into pandas DataFrame
    df = pd.DataFrame([{
        "Age": age,
        "BMI": bmi,
        "Alcohol/Day": alcohol,
        "Gender": gender,
        "Smokes?": smoker,
        "Congestive Heart Failure": chf,
        "Coronary Heart Disease": chd,
        "Heart Attack": heart_attack,
        "Stroke": stroke,
        "Thyroid": thyroid,
        "Race": race
    }])

    # Convert to Spark DataFrame
    sdf = spark.createDataFrame(df)

    # Apply preprocessing
    prepped = tabular_pipeline.transform(sdf)

    # Get model predictions
    rf_preds = rf_model.transform(prepped).select(extract_prob_1(F.col("probability")).alias("rf_prob"))
    gbt_preds = gbt_model.transform(prepped).select(extract_prob_1(F.col("probability")).alias("gbt_prob"))
    lr_preds = lr_model.transform(prepped).select(extract_prob_1(F.col("probability")).alias("lr_prob"))

    # Add row IDs and join
    prepped = prepped.withColumn("row_id", F.monotonically_increasing_id())
    rf_preds = rf_preds.withColumn("row_id", F.monotonically_increasing_id())
    gbt_preds = gbt_preds.withColumn("row_id", F.monotonically_increasing_id())
    lr_preds = lr_preds.withColumn("row_id", F.monotonically_increasing_id())

    ensemble_df = prepped \
        .join(rf_preds, "row_id") \
        .join(gbt_preds, "row_id") \
        .join(lr_preds, "row_id") \
        .withColumn("avg_prob", (F.col("rf_prob") + F.col("gbt_prob") + F.col("lr_prob")) / 3) \
        .withColumn("ensemble_prediction", F.when(F.col("avg_prob") > threshold, 1).otherwise(0).cast("double"))

    result = ensemble_df.select("ensemble_prediction", "avg_prob").collect()[0]
    label = "Likely has cancer" if result["ensemble_prediction"] == 1 else "Unlikely to have cancer"
    confidence = result["avg_prob"]

    # return (label, confidence)
    return f"{label} (Confidence: {confidence:.2f})"



In [5]:
import pickle
import numpy as np
from PIL import Image
from pyspark.sql import Row
from pyspark.ml.linalg import Vectors

def predict_from_image(image):
    # Preprocess and predict using MRI model
    # e.g., PyTorch/TensorFlow logic here
    # prediction = mri_model.predict(image)  # placeholder
    # return float(prediction)
    try:
        # Step 1: Preprocess image just like the dataset pipeline
        img = Image.open(image).convert('L').resize((64, 64))
        arr = np.array(img).flatten().astype(np.float32) / 255.0  # Normalize
        features = arr.reshape(1, -1)  # Match PCA input shape

        # Step 3: Apply PCA transformation
        reduced = images_pca_model.transform(features)
        spark_vector = Vectors.dense(reduced[0])

        # Step 4: Create Spark DataFrame with 'features' column
        row = Row(features=spark_vector)
        spark_df = spark.createDataFrame([row])

        # Step 5: Load Spark RF model and predict
        prediction_df = image_rf_model.transform(spark_df)
        result = prediction_df.select("prediction", "probability").collect()[0]

        label = int(result["prediction"])
        confidence = float(result["probability"][label])  

        return f"{label} (Confidence: {confidence:.2f})"

    except Exception as e:
        print(f"Prediction error: {e}")
        return None

In [6]:
# TODO: Combined prediction
def dual_prediction(image, age, bmi, alcohol, gender, smoker, chf, chd, heart_attack, stroke, thyroid, race):
    image_pred = predict_from_image(image)
    tabular_pred = predict_from_tabular(age, bmi, alcohol, gender, smoker, chf, chd, heart_attack, stroke, thyroid, race)

    # TODO: Figure out how to combine the two predictions and return
    # return (image_pred, tabular_pred)
    

In [14]:
import gradio as gr

iface = gr.Interface(
    fn=predict_from_tabular,
    # fn=dual_prediction,
    inputs=[
        # gr.Image(type="filepath", label="MRI Scan"),
        gr.Radio(["Male", "Female"], label="Gender"), # 1 = Male, 2 = Female
        gr.Slider(18, 80, step=1, label="Age in Years"), # Code corresponds with value
        gr.Radio(["Mexican American", "Other Hispanic", "Non-Hispanic White", "Non-Hispanic Black", "Non-Hispanic Asian", "Other/Multi-Racial"], label="Race"),
        gr.Slider(10.0, 75, step=0.1, label="BMI"), # Code corresponds with value
        gr.Slider(0.0, 14.0, step=0.1, label="Average number of alcoholic drinks a day in the last 12 months"), # Code corresponds with value
        gr.Radio(["Yes", "No"], label="Have you smoked at least 100 cigarettes in your lifetime?"), # 1 = yes, 2 = no -- Applies for the remainder
        gr.Radio(["Yes", "No"], label="Have you ever been diagnosed with Congestive Heart Failure?"),
        gr.Radio(["Yes", "No"], label="Have you ever been diagnosed with Coronary Heart Disease?"),
        gr.Radio(["Yes", "No"], label="Have you ever had a Heart Attack?"),
        gr.Radio(["Yes", "No"], label="Have you ever had a Stroke?"),
        gr.Radio(["Yes", "No"], label="Have you ever had a Thyroid problem?"),
    ],
    outputs="text",
    title="Cancer Risk Predictor",
    description="Enter patient data to predict likelihood of cancer using a Spark-based ensemble model."
)

iface.launch(share=True)


Running on local URL:  http://127.0.0.1:7863
Running on public URL: https://a30e1d39143f8fdf3d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


25/04/29 01:07:36 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
25/04/29 01:07:37 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/04/29 01:07:43 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
25/04/29 01:07:45 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB
                                                                                